In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
%python
container = dbutils.widgets.get("raw")
catalogo = dbutils.widgets.get("catalog_au")
esquema = dbutils.widgets.get("bronze")
storageName = dbutils.widgets.get("adlsproyectofinaletl")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/detalle_ventas.csv"

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-6841757433938230>, line 1
----> 1 container = dbutils.widgets.get("raw")
      2 catalogo = dbutils.widgets.get("catalog_au")
      3 esquema = dbutils.widgets.get("bronze")

File /databricks/python_shell/lib/dbruntime/WidgetHandlerImpl.py:82, in WidgetsHandlerImpl.get(self, name)
     42 def get(self, name: str) -> str:
     43     """ Returns the current value of a widget with the given name.
     44 
     45     :param name: Name of the argument to be accessed
   (...)
     80         ```
     81     """
---> 82     return self._notebookArguments.getArgument(name, self._entry_point.getCurrentBindings())

File /databricks/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py:1362, in JavaMember.__call__(self, *args)
   1356 command = proto.CALL_COMMAND_NAME +\
   1357     self.command_header +\
   1358     args_com

In [0]:
%python
# %python
from pyspark.sql.types import DoubleType

detalleVentas_schema = StructType(fields=[StructField("TipoCompr", StringType(), False),
                                  StructField("NroCompr", StringType(), True),
                                  StructField("CodProd", StringType(), True),
                                  StructField("precio", DoubleType(), True),
                                  StructField("cant", IntegerType(), True),
                                  StructField("subtotal", DoubleType(), True)
])

In [0]:
%python
detalleVentas_df = spark.read \
            .option("header", True) \
            .schema(detalleVentas_schema) \
            .csv(ruta)

In [0]:
detalleVentas_selected_df = detalleVentas_with_timestamp_df.select(col('TipoCompr').alias('tipo_comprobante'), 
                                                   col('NroCompr').alias('nro_comprobante'), 
                                                    col('CodProd').alias('cod_producto')

In [0]:
%python
detalleVentas_selected_df.write.mode('overwrite').saveAsTable(f'{catalogo}.{esquema}.detalle_ventas')